# Truncated-tail canonical map emission

Feed for the `canonicalize-funder-award-ids` job's mapping table
(`openalex.common.funder_award_id_canonical`), per Kyle's 2026-07-29
decision: *"truncated ids: match the distinctive end-part, NO program
recovery."* Design + measurement history: oxjobs
`todo/crossref-award-validation/TRUNCATED-ENDPART-DESIGN.md`.

## Rule

A deposited id that failed registry match maps to a registry award iff its
flattened form (alphanumerics, lowercased) is 6–12 chars, equals the tail of
exactly ONE distinct registry id at the same funder, the registry id is
strictly longer, AND the pair passes the class gates below. Ambiguous tails
are dropped, never guessed.

## Class gates (from the 2026-08-06 blind grade)

A 100-row blind grade of the ungated map (two independent web-verifying
graders) measured 62 CORRECT / 37 WRONG / 1 UNSURE (37% wrong) — far over the <2%-wrong bar, but the
errors were class-structured: bare-digit-only tails at dense numeric-id
funders are dominated by *complete ids of other schemes* (Simons, National
Geographic, Damon Runyon, FP6 contracts, provincial Chinese funders, legacy
NSERC/CIHR/JSPS-era ids). Structured tails graded clean. Emission is
therefore gated to the classes that graded 100% correct:

- **letter-bearing tails** (NIH activity codes, KAKEN, Wellcome, ANR, AHA,
  FCT, NSTC, EPSRC bare-core references…), minus prose/call-name shapes
  (`Harmonia 5`) and DFG signature-reference codes (slash-bearing);
- **NSERC year-serial** (`YYYY-NNNNN` -> `rgpin-YYYY-NNNNN`);
- **NSF leading-zero restoration** (canonical = `'0'` + variant), minus
  date-parsable shapes AND minus variants deposited under >1 funder (the
  Crossref funder-award cross-product artifact: Simons/ERC ids masquerading
  as NSF tails) — **HELD BACK at the 2026-08-06 landing** (class label
  `hold_nsf_zero_restore`): its round-2 carve was post-hoc on the grade
  sample; re-admit only after a fresh prospective grade;
- **DFG 8->9 digit GEPRIS extension**;
- **ISCIII PI-form** (`YY/NNNNN` -> `PIYY/NNNNN`);
- **NIH 6-digit-serial + 2-digit-year** (`NNNNNN-NN`);
- **FAPESP process-shape** (`[YY/]NNNNN-D` check-digit form).

KAKEN 7->8 leading-digit drops looked clean in round 1 (2/2) and were
REFUTED in round 2 (4/4 wrong) — the class is excluded.

Everything else — notably ALL other bare-digit-only tails — is excluded.
Confirmation grade of the gated map: a fresh
100-row sample of the gated map graded 93 CORRECT / 6 WRONG / 1 UNSURE raw;
all 6 wrongs fell in two classes fixed in this notebook (KAKEN 7->8 dropped,
zero-restore cross-funder carve), leaving **93 CORRECT / 0 WRONG / 1 UNSURE
over the emitted population** — under the <2%-wrong bar. With the
zero-restore class held back, the emitted population is **2,937 rows**, all
of it prospectively graded (3,208 minus the 271-row held class).

## Guard interplay

A deposited id that gains a mapping stops being classifiable junk
automatically on the next scoring run (registry hit → verdict change). This
notebook only INSERTs missing (funder_id, variant_id) grains — it never
modifies or deletes existing curation rows, so re-running is safe on any
cadence (recommended: registry-ingest cadence).


In [ ]:
%sql
-- Truncated-tail canonicalization candidates. Rule + gates: see header cell.
-- Registry ambiguity is counted over DISTINCT registry id spellings —
-- duplicate registry ROWS of one spelling (e.g. GTR per-organisation rows)
-- are ONE target. The dev prototype counted rows and wrongly dropped ~250
-- valid recoveries as "ambiguous".
CREATE OR REPLACE TEMPORARY VIEW truncated_tail_candidates AS
WITH dep AS (
  SELECT DISTINCT funder_id, funder_award_id,
         lower(regexp_replace(funder_award_id, '[^0-9A-Za-z]', '')) AS f
  FROM openalex.awards.award_id_verdicts
  WHERE verdict IN ('plausible','garbage')
),
d AS (SELECT * FROM dep WHERE length(f) BETWEEN 6 AND 12),
reg AS (
  SELECT DISTINCT funder_id, funder_award_id AS registry_award_id,
         lower(regexp_replace(funder_award_id, '[^0-9A-Za-z]', '')) AS rf
  FROM openalex.awards.openalex_awards_raw
  WHERE priority >= 3 AND funder_award_id IS NOT NULL
    AND funder_id IN (SELECT DISTINCT funder_id
                      FROM openalex.awards.award_id_verdicts
                      WHERE verdict <> 'unscored')
),
sfx AS (
  SELECT funder_id, registry_award_id, substr(rf, length(rf)-L+1, L) AS tail, L
  FROM reg
  LATERAL VIEW explode(sequence(6,12)) t AS L
  WHERE length(rf) > L
),
pairs AS (
  SELECT DISTINCT d.funder_id, d.funder_award_id, d.f, s.registry_award_id
  FROM d
  JOIN sfx s ON s.funder_id = d.funder_id AND s.tail = d.f AND s.L = length(d.f)
),
uniq AS (
  SELECT funder_id, funder_award_id, f,
         min(registry_award_id) AS registry_award_id
  FROM pairs
  GROUP BY 1, 2, 3
  HAVING count(DISTINCT registry_award_id) = 1
),
xfunder AS (
  -- variant strings deposited under >1 funder: cross-funder collision
  -- evidence (the Crossref funder x award cross-product artifact) — bars
  -- the bare-numeric zero-restore class
  SELECT funder_award_id AS xf_variant
  FROM openalex.awards.award_id_verdicts
  GROUP BY 1
  HAVING count(DISTINCT funder_id) > 1
),
classed AS (
  SELECT uniq.*, (x.xf_variant IS NOT NULL) AS xfunder_deposited,
    CASE
      -- exclusions first (blind-grade wrong classes)
      WHEN funder_id = 4320322511 THEN 'drop_ncn_legacy_registry'     -- old ranking-list entities, superseded by citable UMO registry
      WHEN funder_id = 4320320879 AND funder_award_id LIKE '%/%' THEN 'drop_dfg_signature_code'  -- Geschaeftszeichen, not project-number tails
      WHEN funder_award_id rlike '^[A-Za-z]+ ?[0-9]{1,3}$' THEN 'drop_prose_call_name'           -- "Harmonia 5"-shape call names
      -- keep classes (each graded clean; see design doc)
      WHEN funder_id = 4320334593 AND f rlike '^(19|20)[0-9]{7}$' THEN 'keep_nserc_year_serial'
      WHEN funder_id = 4320306076 AND registry_award_id = concat('0', funder_award_id)
           AND (f rlike '^(0[1-9]|1[0-2])(0[1-9]|[12][0-9]|3[01])[0-9]{2}$'
             OR f rlike '^(0[1-9]|[12][0-9]|3[01])(0[1-9]|1[0-2])[0-9]{2}$'
             OR f rlike '^[0-9]{2}(0[1-9]|1[0-2])(0[1-9]|[12][0-9]|3[01])$') THEN 'drop_date_like'
      WHEN funder_id = 4320306076 AND registry_award_id = concat('0', funder_award_id)
           AND x.xf_variant IS NOT NULL THEN 'drop_xfunder_deposited'
      -- zero-restore HELD BACK at landing 2026-08-06: the class's round-2
      -- carve was post-hoc on the grade sample (design doc "Honesty note");
      -- re-admit via s/hold_/keep_/ only after a fresh prospective grade.
      WHEN funder_id = 4320306076 AND registry_award_id = concat('0', funder_award_id) THEN 'hold_nsf_zero_restore'
      WHEN funder_id = 4320320879 AND f rlike '^[0-9]{8}$'
           AND length(regexp_replace(lower(registry_award_id),'[^0-9a-z]','')) = 9 THEN 'keep_dfg_8to9'
      WHEN funder_id = 4320334764 AND f rlike '^[0-9]{7}$'
           AND registry_award_id rlike '^[0-9]{8}$' THEN 'drop_kaken_7to8_refuted'
      WHEN funder_id = 4320334923 AND funder_award_id rlike '^[0-9]{2}/[0-9]{5}$'
           AND registry_award_id rlike '^[A-Za-z]{2}[0-9]{2}/[0-9]{5}$' THEN 'keep_isciii_pi_form'
      WHEN funder_id = 4320332161 AND funder_award_id rlike '^[0-9]{6}[-‐‒–—][0-9]{2}$' THEN 'keep_nih_serial_year'
      WHEN funder_id = 4320320997 AND funder_award_id rlike '^([0-9]{1,2}/)?[0-9]{5}[-‐‒–—][0-9]$' THEN 'keep_fapesp_process_shape'
      WHEN f rlike '[a-z]' THEN 'keep_letter_bearing'
      ELSE 'drop_bare_numeric'
    END AS cls
  FROM uniq
  LEFT JOIN xfunder x ON x.xf_variant = uniq.funder_award_id
)
SELECT
  funder_id,
  funder_award_id AS variant_id,
  registry_award_id AS canonical_id,
  cls
FROM classed
WHERE cls LIKE 'keep%'


In [ ]:
%sql
-- Mapping table per the canonicalize-funder-award-ids job README sketch:
-- (funder_id, variant_id) -> canonical_id with curation metadata.
-- CREATE IF NOT EXISTS: never clobbers existing curation rows.
-- MUST run before the guard cell — the conflict guard reads this table
-- (first prod run 2026-08-06 failed on the original create-after-guard order).
CREATE TABLE IF NOT EXISTS openalex.common.funder_award_id_canonical (
  funder_id    BIGINT    NOT NULL,
  variant_id   STRING    NOT NULL,
  canonical_id STRING    NOT NULL,
  rule         STRING,
  added_by     STRING,
  added_at     TIMESTAMP,
  notes        STRING
) USING delta


In [0]:
# Count + integrity guards — abort before MERGE if anything is off.
cand = spark.table("truncated_tail_candidates")
n = cand.count()
# Band around the validated build (see notebook header); a collapse to ~0 or
# an explosion means an upstream table changed shape — investigate, don't emit.
assert 2000 <= n <= 8000, f"candidate count {n} outside guard band [2000, 8000]"

grain = cand.groupBy("funder_id", "variant_id").count().filter("count > 1").count()
assert grain == 0, f"{grain} duplicate (funder_id, variant_id) grains"

self_map = cand.filter("variant_id = canonical_id").count()
assert self_map == 0, f"{self_map} self-maps (variant == canonical)"

conflicts = (cand.alias("c")
  .join(spark.table("openalex.common.funder_award_id_canonical").alias("t"),
        ["funder_id", "variant_id"])
  .filter("c.canonical_id <> t.canonical_id").count())
assert conflicts == 0, f"{conflicts} rows conflict with existing curation rows"
print(f"guards pass: {n} candidates")


In [0]:
%sql
-- Idempotent emission: only (funder_id, variant_id) pairs not already in the
-- table are inserted; existing curation rows are never modified.
MERGE INTO openalex.common.funder_award_id_canonical t
USING truncated_tail_candidates c
  ON t.funder_id = c.funder_id AND t.variant_id = c.variant_id
WHEN NOT MATCHED THEN INSERT
  (funder_id, variant_id, canonical_id, rule, added_by, added_at, notes)
VALUES
  (c.funder_id, c.variant_id, c.canonical_id,
   'tail_unique_6_12',
   'crossref-award-validation',
   current_timestamp(),
   'unique-tail truncation recovery vs funder registry; blind-graded 100 rows 2026-08-06')


In [0]:
%sql
-- Post-merge report: what the mapping table now holds, by rule and class
SELECT rule, count(*) AS rows, count(DISTINCT funder_id) AS funders,
       min(added_at) AS first_added, max(added_at) AS last_added
FROM openalex.common.funder_award_id_canonical
GROUP BY rule
